# Trabajo Práctico Final de Machine Learning

## Predicción de calidad del vino mediante Random Forest y selección de atributos con Binary PSO

**Facultad Politécnica — Universidad Nacional de Asunción (FP-UNA)**  
**Asignatura:** Machine Learning  
**Integrantes:** Ivan Renée Paredes Ozuna y Sergio Ramon Gonzalez Silva  
**Problema asignado:** Wine Quality  
**Modelo predictivo:** Random Forest Classifier  
**Metaheurística:** Particle Swarm Optimization binario (Binary PSO)  
**Objetivo de optimización:** Selección de atributos fisicoquímicos  
**Entorno de ejecución:** Google Colab (CPU)

Este notebook es **reproducible** y está diseñado para ejecutarse de principio a fin con *Entorno de ejecución → Ejecutar todo*.  
Incluye un **modo rápido (TEST)** para verificación y un **modo completo (FULL)** para el experimento académico.


---
# 1. Introducción y planteamiento del problema

## 1.1 Contexto

La calidad del vino se determina tradicionalmente mediante evaluación sensorial de expertos, un proceso costoso, subjetivo y difícil de escalar. El dataset *Wine Quality* del repositorio UCI (Cortez et al., 2009) asocia variables fisicoquímicas medibles en laboratorio —acidez, alcohol, sulfatos, densidad, etc.— con una puntuación sensorial de calidad.

Este trabajo aborda la **predicción automática de la calidad** a partir de dichas variables, con énfasis en la **selección de atributos**: no todos los 11 indicadores fisicoquímicos aportan información equivalente, y algunos pueden ser redundantes o ruidosos.

## 1.2 Objetivo general

Investigar si la selección de atributos mediante **Binary PSO** permite obtener un **Random Forest Classifier** más eficiente (menos variables) que mantenga o mejore el rendimiento predictivo respecto al modelo que utiliza las 11 características.

## 1.3 Objetivos específicos

1. Caracterizar el dataset de vinos tintos y justificar el planteamiento de clasificación.
2. Construir un *baseline* Random Forest con los 11 atributos y hiperparámetros fijos.
3. Formular e implementar Binary PSO para búsqueda de subconjuntos de atributos.
4. Comparar, con protocolo experimental controlado, el modelo base frente al modelo con atributos seleccionados.
5. Analizar convergencia, estabilidad entre semillas y costo computacional.
6. Exportar evidencia reproducible (CSV, gráficos y tablas) para artículo IEEE y póster científico.

## 1.4 Random Forest (visión metodológica)

Random Forest (Breiman, 2001) es un ensamble de árboles de decisión entrenados sobre *bootstrap samples*, con aleatorización de atributos en cada división. Ofrece buen desempeño en datos tabulares, robustez ante outliers relativos y estimación implícita de importancia de variables. En este trabajo **no se optimizan sus hiperparámetros**: se fijan de forma justificada para aislar el efecto de la selección de atributos.

## 1.5 Binary PSO

Particle Swarm Optimization (Kennedy y Eberhart, 1995) es una metaheurística inspirada en el comportamiento social de enjambres. En la variante **binaria**, cada partícula es un vector de bits que indica inclusión/exclusión de cada atributo. El enjambre explora el espacio de \(2^{11} = 2048\) subconjuntos posibles, guiado por memorias individual y colectiva.

## 1.6 Justificación de la selección de atributos

Con 11 variables, el espacio es enumerable en teoría, pero evaluar cada subconjunto con validación cruzada y un ensamble es costoso. Además, el enunciado del trabajo asigna específicamente Binary PSO. La selección de atributos puede:

- reducir dimensionalidad y complejidad del modelo;
- atenuar ruido y colinealidad;
- mejorar interpretabilidad (qué variables fisicoquímicas importan);
- eventualmente mejorar o preservar métricas de clasificación.

Reducir atributos disminuye el espacio de hipótesis efectivo del ensamble, puede acelerar el entrenamiento/inferencia y facilita la interpretación fisicoquímica del modelo.

## 1.7 Preguntas de investigación

1. ¿Binary PSO encuentra subconjuntos con rendimiento comparable o superior al de las 11 variables?
2. ¿Qué atributos se seleccionan de forma estable entre ejecuciones independientes?
3. ¿El presupuesto de ~30 evaluaciones de función objetivo es suficiente para observar convergencia útil?
4. ¿El costo de la búsqueda se justifica frente al ahorro de variables en el modelo final?

## 1.8 Hipótesis experimental

**H1:** Existe al menos un subconjunto propio de atributos cuyo Random Forest alcanza un F1-macro (CV) no inferior al baseline, con menos de 11 variables.  
**H0 (operativa):** Bajo el presupuesto y protocolo definidos, Binary PSO no mejora el compromiso desempeño–parsimonia respecto al modelo completo.

Es metodológicamente válido que H1 no se confirme; el valor científico reside en la evidencia experimental.


---
# 2. Configuración del entorno

Se instalan dependencias compatibles con Google Colab (CPU), se registran versiones, se fijan semillas y se crean carpetas de salida.  
El parámetro `EXPERIMENT_MODE` controla el presupuesto:

| Modo | Uso | Presupuesto aproximado |
|------|-----|------------------------|
| `TEST` | Verificar que el pipeline funciona | Reducido (pocas evaluaciones y 1 semilla) |
| `FULL` | Resultados académicos | ~30 evaluaciones/ejecución × varias semillas |

**Importante:** los resultados del modo `TEST` **no** deben presentarse como definitivos.


In [ ]:
# =============================================================================
# Celda de control experimental
# =============================================================================
# Cambiar a "FULL" para el experimento completo del informe académico.
EXPERIMENT_MODE = "TEST"  # opciones: "TEST" | "FULL"

assert EXPERIMENT_MODE in {"TEST", "FULL"}, "EXPERIMENT_MODE debe ser TEST o FULL"
print(f"Modo de experimento seleccionado: {EXPERIMENT_MODE}")


In [ ]:
# Instalación de dependencias (idempotente en Colab / local)
import sys
import subprocess

PACKAGES = [
    "pandas",
    "numpy",
    "matplotlib",
    "seaborn",
    "scikit-learn",
    "pyswarms",
    "scipy",
]

def _pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

try:
    import pyswarms  # noqa: F401
except Exception:
    print("Instalando dependencias...")
    _pip_install(PACKAGES)
    print("Instalación finalizada.")
else:
    missing = []
    for name, mod in [
        ("pandas", "pandas"),
        ("numpy", "numpy"),
        ("matplotlib", "matplotlib"),
        ("seaborn", "seaborn"),
        ("scikit-learn", "sklearn"),
        ("scipy", "scipy"),
    ]:
        try:
            __import__(mod)
        except Exception:
            missing.append(name)
    if missing:
        print(f"Instalando faltantes: {missing}")
        _pip_install(missing)
    print("Dependencias disponibles.")


In [ ]:
# Importaciones organizadas
from __future__ import annotations

import json
import os
import platform
import random
import time
import warnings
from collections import Counter
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from pyswarms.discrete.binary import BinaryPSO

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

print("Importaciones OK.")


In [ ]:
# Registro de versiones y entorno
import sklearn

def _pkg_version(name: str) -> str:
    try:
        mod = __import__(name)
        return getattr(mod, "__version__", "desconocida")
    except Exception as exc:
        return f"error: {exc}"

ENV_INFO = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "numpy": _pkg_version("numpy"),
    "pandas": _pkg_version("pandas"),
    "matplotlib": _pkg_version("matplotlib"),
    "seaborn": _pkg_version("seaborn"),
    "sklearn": sklearn.__version__,
    "scipy": _pkg_version("scipy"),
    "pyswarms": _pkg_version("pyswarms"),
}

print("=== Versiones del entorno ===")
for k, v in ENV_INFO.items():
    print(f"{k:12s}: {v}")

try:
    import multiprocessing as mp
    N_CPUS = mp.cpu_count()
except Exception:
    N_CPUS = os.cpu_count() or 1

print(f"\nCPUs detectadas: {N_CPUS}")
print("Este experimento está diseñado para CPU; no se requiere GPU.")


In [ ]:
# Semillas globales y carpetas de resultados
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

# En Colab: /content/resultados ; en local: ./resultados
if Path("/content").exists():
    BASE_DIR = Path("/content/resultados")
else:
    BASE_DIR = Path("./resultados")

DIRS = {
    "root": BASE_DIR,
    "csv": BASE_DIR / "csv",
    "figures": BASE_DIR / "figures",
    "eda": BASE_DIR / "figures" / "eda",
    "convergence": BASE_DIR / "figures" / "convergence",
    "evaluation": BASE_DIR / "figures" / "evaluation",
    "logs": BASE_DIR / "logs",
}

for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

print("Directorios creados:")
for name, path in DIRS.items():
    print(f"  {name:12s} -> {path.resolve()}")

with open(DIRS["logs"] / "entorno.json", "w", encoding="utf-8") as f:
    json.dump({**ENV_INFO, "n_cpus": N_CPUS, "mode": EXPERIMENT_MODE}, f, indent=2)


---
# 3. Análisis exploratorio de datos (EDA)

## Origen y justificación del dataset

Se utiliza **winequality-red.csv** del *UCI Machine Learning Repository* (Cortez et al., 2009):

- **Fuente:** https://archive.ics.uci.edu/dataset/186/wine+quality  
- **URL directa del CSV:** `https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv`  
- **Licencia / uso:** dataset público de investigación (UCI); citar a los autores originales.  
- **Contenido:** 11 atributos fisicoquímicos + variable `quality` (puntuación sensorial entera).

### ¿Por qué solo vinos tintos?

Existen dos archivos (tinto y blanco). Mezclarlos introduce un factor de **tipo de vino** no etiquetado de forma homogénea en un único CSV, y las distribuciones fisicoquímicas difieren entre variedades. Trabajar únicamente con tintos:

1. respeta el número de atributos asignado (11);
2. evita confusión de poblaciones;
3. mantiene el diseño experimental comparable al enunciado del grupo.

## Prevención de fuga de información

El EDA descriptivo global se presenta para caracterizar el dataset, pero **toda decisión de modelado** (transformaciones, umbral de clase, selección PSO) se basa exclusivamente en el conjunto de **desarrollo** tras separar el test. El test permanece reservado hasta la Sección 11.


In [ ]:
# Carga automática y reproducible del dataset
DATA_URL = (
    "https://archive.ics.uci.edu/ml/machine-learning-databases/"
    "wine-quality/winequality-red.csv"
)
LOCAL_CACHE = DIRS["root"] / "winequality-red.csv"

def load_wine_quality(url: str = DATA_URL, cache_path: Path = LOCAL_CACHE) -> pd.DataFrame:
    """Descarga (o lee caché) el CSV de vinos tintos UCI."""
    try:
        df = pd.read_csv(url, sep=";")
        df.to_csv(cache_path, index=False)
        print(f"Dataset descargado desde UCI y cacheado en: {cache_path}")
    except Exception as primary_err:
        print(f"Fallo descarga primaria: {primary_err}")
        alt_url = (
            "https://raw.githubusercontent.com/jbrownlee/Datasets/master/winequality-red.csv"
        )
        try:
            cols = [
                "fixed acidity", "volatile acidity", "citric acid", "residual sugar",
                "chlorides", "free sulfur dioxide", "total sulfur dioxide", "density",
                "pH", "sulphates", "alcohol", "quality",
            ]
            df = pd.read_csv(alt_url, header=None, names=cols)
            df.to_csv(cache_path, index=False)
            print(f"Dataset cargado desde mirror alternativo: {alt_url}")
        except Exception as secondary_err:
            if cache_path.exists():
                print("Usando caché local previa.")
                df = pd.read_csv(cache_path)
            else:
                raise RuntimeError(
                    f"No se pudo cargar el dataset. Errores: {primary_err} | {secondary_err}"
                )
    return df

df_raw = load_wine_quality()
print(f"Dimensiones: {df_raw.shape}")
df_raw.head()


In [ ]:
# Dimensiones, tipos y estadísticas descriptivas
print("=== Información general ===")
print(f"Filas: {df_raw.shape[0]} | Columnas: {df_raw.shape[1]}")
print("\nTipos de datos:")
print(df_raw.dtypes)
print("\nEstadísticas descriptivas:")
display_df = df_raw.describe().T
display_df["skew"] = df_raw.skew(numeric_only=True)
display_df["kurtosis"] = df_raw.kurtosis(numeric_only=True)
display(display_df.round(4))

FEATURE_COLS = [c for c in df_raw.columns if c != "quality"]
TARGET_COL = "quality"
assert len(FEATURE_COLS) == 11, f"Se esperaban 11 atributos; hay {len(FEATURE_COLS)}"
print("\nAtributos predictivos:")
for i, c in enumerate(FEATURE_COLS, 1):
    print(f"  {i:2d}. {c}")


In [ ]:
# Valores faltantes, duplicados e inconsistencias
missing = df_raw.isna().sum()
print("=== Valores faltantes ===")
print(missing)
print(f"Total de celdas NA: {int(missing.sum())}")

n_dup = int(df_raw.duplicated().sum())
print(f"\n=== Duplicados exactos ===\nCantidad: {n_dup} ({100 * n_dup / len(df_raw):.2f}%)")

print("\n=== Chequeos de inconsistencia ===")
print(f"quality fuera de [0,10]: {((df_raw['quality'] < 0) | (df_raw['quality'] > 10)).sum()}")
print(f"valores negativos en features: {(df_raw[FEATURE_COLS] < 0).sum().sum()}")
print(f"pH fuera de [0,14]: {((df_raw['pH'] < 0) | (df_raw['pH'] > 14)).sum()}")
print(f"density fuera de [0.9, 1.1]: {((df_raw['density'] < 0.9) | (df_raw['density'] > 1.1)).sum()}")

print(
    "\nDecisión de tratamiento:\n"
    "- No se eliminan duplicados automáticamente: pueden corresponder a mediciones\n"
    "  repetidas legítimas o a vinos con el mismo perfil fisicoquímico; eliminarlos\n"
    "  podría sesgar la distribución de clases.\n"
    "- No hay valores faltantes; no se imputa.\n"
    "- No se eliminan outliers a priori: Random Forest es relativamente robusto y\n"
    "  los extremos pueden ser vinos de calidad atípica informativos.\n"
)


In [ ]:
# Separación temprana train/test para evitar fuga en decisiones posteriores
X_all = df_raw[FEATURE_COLS].copy()
y_quality = df_raw[TARGET_COL].astype(int).copy()

X_dev_raw, X_test_raw, y_dev_quality, y_test_quality = train_test_split(
    X_all,
    y_quality,
    test_size=0.20,
    random_state=GLOBAL_SEED,
    stratify=y_quality,
)

print(f"Desarrollo: {X_dev_raw.shape[0]} muestras ({100*X_dev_raw.shape[0]/len(df_raw):.1f}%)")
print(f"Test:       {X_test_raw.shape[0]} muestras ({100*X_test_raw.shape[0]/len(df_raw):.1f}%)")
print("El conjunto de test queda RESERVADO hasta la evaluación final.")


In [ ]:
# Distribuciones de variables (EDA sobre desarrollo)
fig, axes = plt.subplots(3, 4, figsize=(14, 10))
axes = axes.ravel()
for i, col in enumerate(FEATURE_COLS):
    axes[i].hist(X_dev_raw[col], bins=30, color="#2c7fb8", edgecolor="white", alpha=0.85)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Frecuencia")
axes[-1].axis("off")
fig.suptitle("Distribuciones de atributos fisicoquímicos (conjunto de desarrollo)", fontsize=14)
fig.tight_layout()
fig.savefig(DIRS["eda"] / "01_distribuciones_atributos.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Asimetría, dispersión y outliers (IQR) — solo desarrollo
def iqr_outlier_counts(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in frame.columns:
        q1, q3 = frame[col].quantile(0.25), frame[col].quantile(0.75)
        iqr = q3 - q1
        low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = int(((frame[col] < low) | (frame[col] > high)).sum())
        rows.append({
            "atributo": col,
            "media": frame[col].mean(),
            "std": frame[col].std(),
            "skew": frame[col].skew(),
            "kurtosis": frame[col].kurtosis(),
            "n_outliers_iqr": n_out,
            "pct_outliers_iqr": 100 * n_out / len(frame),
        })
    return pd.DataFrame(rows)

outlier_summary = iqr_outlier_counts(X_dev_raw)
display(outlier_summary.round(3))
outlier_summary.to_csv(DIRS["csv"] / "eda_asimetria_outliers_dev.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=X_dev_raw, orient="h", ax=ax, color="#7fcdbb")
ax.set_title("Boxplots de atributos (desarrollo) — detección visual de extremos")
ax.set_xlabel("Valor")
fig.tight_layout()
fig.savefig(DIRS["eda"] / "02_boxplots_atributos.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Matriz de correlaciones y redundancia potencial (desarrollo)
corr = X_dev_raw.corr(method="pearson")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax, square=True)
ax.set_title("Matriz de correlación de Pearson (desarrollo)")
fig.tight_layout()
fig.savefig(DIRS["eda"] / "03_correlacion_atributos.png", dpi=150, bbox_inches="tight")
plt.show()

pairs = []
cols = corr.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr.iloc[i, j]
        if abs(r) >= 0.70:
            pairs.append((cols[i], cols[j], float(r)))

print("Pares con |correlación| >= 0.70 (candidatos a redundancia):")
if not pairs:
    print("  Ninguno supera el umbral 0.70.")
else:
    for a, b, r in sorted(pairs, key=lambda t: -abs(t[2])):
        print(f"  {a}  <->  {b}  :  r = {r:.3f}")

dev_tmp = X_dev_raw.copy()
dev_tmp["quality"] = y_dev_quality.values
corr_target = (
    dev_tmp.corr(method="spearman")["quality"]
    .drop("quality")
    .reindex(dev_tmp.corr(method="spearman")["quality"].drop("quality").abs().sort_values(ascending=False).index)
)
print("\nCorrelación de Spearman con quality (desarrollo):")
print(corr_target.round(3))


In [ ]:
# Distribución de quality y desbalance
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
vc = y_dev_quality.value_counts().sort_index()
axes[0].bar(vc.index.astype(str), vc.values, color="#2c7fb8", edgecolor="white")
axes[0].set_title("Distribución de quality (desarrollo)")
axes[0].set_xlabel("quality")
axes[0].set_ylabel("Frecuencia")
for x, v in zip(vc.index.astype(str), vc.values):
    axes[0].text(x, v + 5, str(v), ha="center", fontsize=9)

pct = 100 * vc / vc.sum()
axes[1].bar(pct.index.astype(str), pct.values, color="#fdae61", edgecolor="white")
axes[1].set_title("Porcentaje por clase de quality (desarrollo)")
axes[1].set_xlabel("quality")
axes[1].set_ylabel("%")
fig.tight_layout()
fig.savefig(DIRS["eda"] / "04_distribucion_quality.png", dpi=150, bbox_inches="tight")
plt.show()

print("Conteo quality (desarrollo):")
print(vc)
print("\nProporciones:")
print((vc / vc.sum()).round(4))
print(f"\nÍndice de desbalance (max/min): {(vc.max() / max(vc.min(), 1)):.2f}")


In [ ]:
# Relaciones preliminares atributo-calidad (desarrollo)
top_feats = corr_target.head(4).index.tolist()
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), top_feats):
    sns.boxplot(x=y_dev_quality, y=X_dev_raw[col], ax=ax, color="#a6bddb")
    ax.set_title(f"{col} vs quality")
    ax.set_xlabel("quality")
    ax.set_ylabel(col)
fig.suptitle("Relaciones preliminares (4 atributos más asociados a quality)", fontsize=13)
fig.tight_layout()
fig.savefig(DIRS["eda"] / "05_relaciones_atributo_quality.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 4. Definición del problema de clasificación

## Alternativas consideradas

| Enfoque | Definición | Ventajas | Limitaciones |
|---------|------------|----------|--------------|
| **Multiclase** | Predecir `quality` original | Conserva granularidad sensorial | Clases extremas muy raras; métricas inestables; CV frágil |
| **Binaria** | Umbral sobre `quality` | Mejor soporte por clase; métricas más estables; encaje con RF + PSO bajo presupuesto | Pierde matices ordinales |

## Justificación del umbral

En el conjunto de desarrollo, las clases se concentran en 5 y 6. Un umbral natural en la literatura de Wine Quality es:

\[
y = \begin{cases}
1 & \text{si } quality \ge 6 \quad \text{(calidad aceptable/alta)} \\
0 & \text{si } quality \le 5 \quad \text{(calidad baja/media-baja)}
\end{cases}
\]

Este corte:

1. produce dos clases con soporte suficiente para CV estratificada de 3 folds;
2. es interpretable en un contexto de control de calidad;
3. es consistente con numerosos trabajos sobre este dataset.

**Limitación:** vinos con `quality = 6` y `quality = 8` quedan en la misma clase; el modelo no distingue excelencia de mera aceptabilidad.

## Métricas

- **Principal:** **F1-macro** — promedio no ponderado del F1 por clase; penaliza mal desempeño en la clase minoritaria.
- **Secundarias:** balanced accuracy, precision/recall (por clase y macro), accuracy, ROC-AUC (binario).

No se usa accuracy como métrica principal por el desbalance residual.


In [ ]:
# Análisis del umbral y definición de etiquetas binarias
QUALITY_THRESHOLD = 6

def binarize_quality(y, threshold: int = QUALITY_THRESHOLD) -> np.ndarray:
    y = np.asarray(y)
    return (y >= threshold).astype(int)

print("=== Balance bajo distintos umbrales (solo desarrollo) ===")
for thr in [5, 6, 7]:
    yb = binarize_quality(y_dev_quality, thr)
    pos = yb.mean()
    print(
        f"threshold >= {thr}: clase 1 = {pos:.3f} | clase 0 = {1-pos:.3f} "
        f"| n1={yb.sum()} n0={(1-yb).sum()}"
    )

y_dev = binarize_quality(y_dev_quality)
y_test = binarize_quality(y_test_quality)

print(f"\nUmbral adoptado: quality >= {QUALITY_THRESHOLD}")
print(f"Desarrollo — clase 0: {(y_dev == 0).sum()} | clase 1: {(y_dev == 1).sum()}")
print(f"Test       — clase 0: {(y_test == 0).sum()} | clase 1: {(y_test == 1).sum()}")

fig, ax = plt.subplots(figsize=(6, 4))
counts = pd.Series(y_dev).value_counts().sort_index()
ax.bar(["Baja (0)", "Alta (1)"], counts.values, color=["#d73027", "#1a9850"], edgecolor="white")
ax.set_title(f"Distribución binaria (desarrollo), umbral quality >= {QUALITY_THRESHOLD}")
ax.set_ylabel("Frecuencia")
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha="center")
fig.tight_layout()
fig.savefig(DIRS["eda"] / "06_distribucion_binaria.png", dpi=150, bbox_inches="tight")
plt.show()

PRIMARY_METRIC = "f1_macro"
SECONDARY_METRICS = ["balanced_accuracy", "precision_macro", "recall_macro", "accuracy", "roc_auc"]
print(f"Métrica principal: {PRIMARY_METRIC}")
print(f"Métricas secundarias: {SECONDARY_METRICS}")


---
# 5. Preparación de los datos

- **X / y:** atributos fisicoquímicos y etiqueta binaria.
- **Split:** 80% desarrollo / 20% test, estratificado (ya realizado).
- **CV:** `StratifiedKFold` con 3 particiones sobre desarrollo.
- **Transformaciones:** `StandardScaler` dentro de un `Pipeline` (ajustado solo en folds de entrenamiento).
- **Test:** no se usa para PSO, ni para elegir semillas, ni para tuning.

Random Forest no requiere escalado estrictamente, pero el pipeline documenta buenas prácticas y facilita extensiones futuras sin cambiar el protocolo.


In [ ]:
# Matrices finales y protocolo de validación
X_dev = X_dev_raw.reset_index(drop=True)
X_test = X_test_raw.reset_index(drop=True)
y_dev = np.asarray(y_dev)
y_test = np.asarray(y_test)

N_FEATURES = X_dev.shape[1]
N_SPLITS = 3
CV = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=GLOBAL_SEED)

# Hiperparámetros FIJOS del Random Forest (no se optimizan)
RF_PARAMS = dict(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
)

def make_rf(random_state: int = GLOBAL_SEED) -> RandomForestClassifier:
    return RandomForestClassifier(random_state=random_state, **RF_PARAMS)

def make_pipeline(random_state: int = GLOBAL_SEED) -> Pipeline:
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("clf", make_rf(random_state=random_state)),
        ]
    )

print(f"X_dev={X_dev.shape}, X_test={X_test.shape}, n_features={N_FEATURES}")
print(f"CV: StratifiedKFold(n_splits={N_SPLITS}, shuffle=True, random_state={GLOBAL_SEED})")
print("Hiperparámetros RF fijos:", RF_PARAMS)


---
# 6. Modelo de referencia (Baseline)

Se entrena/evalúa un Random Forest con **las 11 características** mediante la misma CV de 3 folds que usará PSO.  
Los hiperparámetros permanecen constantes en todo el notebook para que la comparación isole el efecto de la selección de atributos.

Justificación breve de la configuración:

- `n_estimators=100`: compromiso estabilidad/tiempo en CPU.
- `max_depth=12`: limita sobreajuste en dataset mediano.
- `min_samples_leaf=2`: regularización suave.
- `max_features="sqrt"`: recomendación clásica de Breiman para clasificación.


In [ ]:
def evaluate_feature_subset(
    X: pd.DataFrame,
    y: np.ndarray,
    feature_mask,
    cv: StratifiedKFold,
    random_state: int = GLOBAL_SEED,
) -> Dict[str, Any]:
    """Evalúa un subconjunto de atributos con CV estratificada. No usa test."""
    mask = np.asarray(feature_mask).astype(bool)
    n_selected = int(mask.sum())
    if n_selected == 0:
        return {
            "valid": False,
            "n_features": 0,
            "f1_macro_mean": 0.0,
            "f1_macro_std": 0.0,
            "balanced_accuracy_mean": 0.0,
            "precision_macro_mean": 0.0,
            "recall_macro_mean": 0.0,
            "accuracy_mean": 0.0,
            "roc_auc_mean": 0.0,
            "fit_time_mean": 0.0,
            "score_time_mean": 0.0,
            "total_time": 0.0,
            "n_model_fits": 0,
        }

    X_sub = X.loc[:, X.columns[mask]]
    pipe = make_pipeline(random_state=random_state)
    scoring = {
        "f1_macro": "f1_macro",
        "balanced_accuracy": "balanced_accuracy",
        "precision_macro": "precision_macro",
        "recall_macro": "recall_macro",
        "accuracy": "accuracy",
        "roc_auc": "roc_auc",
    }
    t0 = time.perf_counter()
    cv_res = cross_validate(
        pipe,
        X_sub,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=False,
    )
    total_time = time.perf_counter() - t0

    out = {
        "valid": True,
        "n_features": n_selected,
        "total_time": total_time,
        "n_model_fits": N_SPLITS,
        "fit_time_mean": float(np.mean(cv_res["fit_time"])),
        "score_time_mean": float(np.mean(cv_res["score_time"])),
    }
    for key in scoring:
        out[f"{key}_mean"] = float(np.mean(cv_res[f"test_{key}"]))
        out[f"{key}_std"] = float(np.std(cv_res[f"test_{key}"]))
    return out


baseline_mask = np.ones(N_FEATURES, dtype=int)
t_base0 = time.perf_counter()
baseline_cv = evaluate_feature_subset(X_dev, y_dev, baseline_mask, CV, random_state=GLOBAL_SEED)
baseline_cv["wall_time"] = time.perf_counter() - t_base0

print("=== Baseline Random Forest (11 atributos, CV 3-fold) ===")
for k in [
    "f1_macro_mean", "f1_macro_std", "balanced_accuracy_mean",
    "precision_macro_mean", "recall_macro_mean", "accuracy_mean",
    "roc_auc_mean", "n_features", "total_time", "n_model_fits",
]:
    print(f"{k:28s}: {baseline_cv[k]}")

baseline_row = {
    "modelo": "RF_baseline_11",
    "n_features": baseline_cv["n_features"],
    "f1_macro_cv": baseline_cv["f1_macro_mean"],
    "f1_macro_cv_std": baseline_cv["f1_macro_std"],
    "balanced_accuracy_cv": baseline_cv["balanced_accuracy_mean"],
    "roc_auc_cv": baseline_cv["roc_auc_mean"],
    "tiempo_s": baseline_cv["total_time"],
}
pd.DataFrame([baseline_row]).to_csv(DIRS["csv"] / "baseline_cv.csv", index=False)


---
# 7. Formulación matemática de Binary PSO

## Representación

Cada partícula es un vector binario de dimensión \(d = 11\):

\[
\mathbf{x} = [x_1, x_2, \ldots, x_{11}], \quad x_j \in \{0,1\}
\]

donde \(x_j = 1\) indica que el atributo \(j\) se incluye en el Random Forest.

## Espacio de búsqueda

\[
\mathcal{S} = \{0,1\}^{11}, \quad |\mathcal{S}| = 2^{11} = 2048
\]

## Restricción de factibilidad

No se permite entrenar con cero atributos:

\[
\sum_{j=1}^{11} x_j \ge 1
\]

Las soluciones inviables reciben un costo alto sin entrenar el modelo.

## Función objetivo (minimización)

\[
C(\mathbf{x}) = \bigl(1 - \mathrm{F1\_macro\_CV}(\mathbf{x})\bigr) + \lambda \cdot \frac{n(\mathbf{x})}{11}
\]

donde \(n(\mathbf{x}) = \sum_j x_j\) y \(\lambda > 0\) es una penalización de parsimonia.

**Sentido:** minimizar \(C\) equivale a maximizar F1-macro de CV y, en segunda prioridad, preferir subconjuntos más pequeños.

### Justificación de \(\lambda\)

Se adopta \(\lambda = 0.05\). Bajo este peso, reducir un atributo mejora el costo en \(0.05/11 \approx 0.0045\), equivalente a tolerar una caída de F1 de ~0.45 puntos porcentuales a cambio de eliminar una variable. Así, la parsimonia solo "gana" si el desempeño se mantiene prácticamente estable.


In [ ]:
LAMBDA_PENALTY = 0.05  # penalización de parsimonia justificada en la Sección 7
INVALID_COST = 2.0

def objective_cost(f1_macro: float, n_selected: int, lam: float = LAMBDA_PENALTY) -> float:
    """Costo a minimizar: (1 - F1) + λ * (n/11)."""
    return (1.0 - float(f1_macro)) + lam * (n_selected / N_FEATURES)

c_base = objective_cost(baseline_cv["f1_macro_mean"], 11)
print(f"λ = {LAMBDA_PENALTY}")
print(f"Costo del baseline (11 attrs, F1={baseline_cv['f1_macro_mean']:.4f}): {c_base:.6f}")
print("Una solución inválida (0 attrs) se penalizará con costo = 2.0 (sin entrenar RF).")


---
# 8. Implementación de Binary PSO

Se utiliza `pyswarms.discrete.binary.BinaryPSO`.

## Presupuesto de evaluaciones

En PySwarms, la función objetivo recibe **todo el enjambre** en cada iteración. Por lo tanto:

\[
N_{\mathrm{evaluaciones}} \approx n_{\mathrm{particles}} \times n_{\mathrm{iters}}
\]

Diseño para el modo FULL (referencia del enunciado ≈ 30 evaluaciones):

- `n_particles = 6`
- `iters = 5`
- ⇒ \(6 \times 5 = 30\) evaluaciones de función objetivo por ejecución

Modo TEST: presupuesto reducido para verificar el pipeline.

Cada evaluación individual (cada partícula) se registra en un log, incluyendo repeticiones e inviables.


In [ ]:
# Configuración de presupuesto según modo
if EXPERIMENT_MODE == "FULL":
    PSO_N_PARTICLES = 6
    PSO_ITERS = 5
    PSO_SEEDS = [42, 7, 123, 2024, 99]  # 5 ejecuciones independientes
else:
    PSO_N_PARTICLES = 3
    PSO_ITERS = 2
    PSO_SEEDS = [42]  # una sola semilla para verificación

PSO_OPTIONS = {
    "c1": 0.5,   # cognitivo
    "c2": 0.5,   # social
    "w": 0.9,    # inercia
    "k": min(3, PSO_N_PARTICLES - 1) if PSO_N_PARTICLES > 1 else 1,
    "p": 2,      # norma Minkowski (Euclídea)
}

EXPECTED_EVALS = PSO_N_PARTICLES * PSO_ITERS

print("=== Configuración Binary PSO ===")
print(f"Modo              : {EXPERIMENT_MODE}")
print(f"Partículas        : {PSO_N_PARTICLES}")
print(f"Iteraciones       : {PSO_ITERS}")
print(f"Evaluaciones esp. : {EXPECTED_EVALS}")
print(f"Semillas          : {PSO_SEEDS}")
print(f"Opciones          : {PSO_OPTIONS}")
print("Criterio de parada : número fijo de iteraciones (presupuesto cerrado).")


In [ ]:
@dataclass
class EvaluationLog:
    """Registro de todas las evaluaciones de la función objetivo."""
    rows: List[Dict[str, Any]] = field(default_factory=list)
    eval_counter: int = 0

    def reset(self) -> None:
        self.rows.clear()
        self.eval_counter = 0

    def as_dataframe(self) -> pd.DataFrame:
        return pd.DataFrame(self.rows)


class FeatureSelectionObjective:
    """Función objetivo vectorizada para BinaryPSO + logging detallado."""

    def __init__(
        self,
        X: pd.DataFrame,
        y: np.ndarray,
        cv: StratifiedKFold,
        seed: int,
        lam: float = LAMBDA_PENALTY,
        cache: bool = True,
    ):
        self.X = X
        self.y = y
        self.cv = cv
        self.seed = seed
        self.lam = lam
        self.cache = cache
        self.log = EvaluationLog()
        self._cache: Dict[Tuple[int, ...], Dict[str, Any]] = {}
        self.effective_trains = 0

    def _mask_key(self, mask: np.ndarray) -> Tuple[int, ...]:
        return tuple(int(v) for v in mask.tolist())

    def evaluate_one(self, mask: np.ndarray) -> Dict[str, Any]:
        key = self._mask_key(mask)
        n_sel = int(mask.sum())
        t0 = time.perf_counter()

        if n_sel == 0:
            result = {
                "valid": False,
                "status": "invalid_zero_features",
                "n_features": 0,
                "f1_macro": 0.0,
                "cost": INVALID_COST,
                "from_cache": False,
                "trained": False,
            }
            result["time_s"] = time.perf_counter() - t0
            return result

        if self.cache and key in self._cache:
            cached = self._cache[key]
            result = {
                **cached,
                "from_cache": True,
                "trained": False,
                "status": "cache_hit",
            }
            result["time_s"] = time.perf_counter() - t0
            return result

        metrics = evaluate_feature_subset(
            self.X, self.y, mask, self.cv, random_state=self.seed
        )
        cost = objective_cost(metrics["f1_macro_mean"], n_sel, self.lam)
        result = {
            "valid": True,
            "status": "trained",
            "n_features": n_sel,
            "f1_macro": metrics["f1_macro_mean"],
            "f1_macro_std": metrics["f1_macro_std"],
            "balanced_accuracy": metrics["balanced_accuracy_mean"],
            "roc_auc": metrics["roc_auc_mean"],
            "cost": cost,
            "from_cache": False,
            "trained": True,
            "cv_time_s": metrics["total_time"],
            "n_model_fits": metrics["n_model_fits"],
        }
        result["time_s"] = time.perf_counter() - t0
        self.effective_trains += 1
        if self.cache:
            self._cache[key] = {
                k: result[k]
                for k in [
                    "valid", "n_features", "f1_macro", "f1_macro_std",
                    "balanced_accuracy", "roc_auc", "cost",
                ]
            }
            self._cache[key]["status"] = "trained"
        return result

    def __call__(self, swarm_positions: np.ndarray) -> np.ndarray:
        """PySwarms pasa un array (n_particles, n_dimensions)."""
        costs = []
        positions = (swarm_positions >= 0.5).astype(int)
        for particle in positions:
            self.log.eval_counter += 1
            ev = self.log.eval_counter
            res = self.evaluate_one(particle)
            selected = [FEATURE_COLS[i] for i, b in enumerate(particle) if b == 1]
            row = {
                "seed": self.seed,
                "eval_id": ev,
                "selection_binary": "b" + "".join(str(int(b)) for b in particle),
                "n_features": res["n_features"],
                "f1_macro": res.get("f1_macro", 0.0),
                "cost": res["cost"],
                "time_s": res["time_s"],
                "status": res["status"],
                "from_cache": res.get("from_cache", False),
                "trained": res.get("trained", False),
                "balanced_accuracy": res.get("balanced_accuracy", np.nan),
                "roc_auc": res.get("roc_auc", np.nan),
                "selected_features": "|".join(selected),
            }
            self.log.rows.append(row)
            costs.append(res["cost"])
        return np.asarray(costs, dtype=float)


def run_binary_pso(seed: int) -> Dict[str, Any]:
    """Ejecuta una corrida independiente de Binary PSO."""
    np.random.seed(seed)
    random.seed(seed)

    objective = FeatureSelectionObjective(
        X_dev, y_dev, CV, seed=seed, lam=LAMBDA_PENALTY
    )

    optimizer = BinaryPSO(
        n_particles=PSO_N_PARTICLES,
        dimensions=N_FEATURES,
        options=PSO_OPTIONS,
    )

    t0 = time.perf_counter()
    best_cost, best_pos = optimizer.optimize(
        objective,
        iters=PSO_ITERS,
        verbose=False,
    )
    total_time = time.perf_counter() - t0

    best_mask = (np.asarray(best_pos) >= 0.5).astype(int)
    best_metrics = evaluate_feature_subset(
        X_dev, y_dev, best_mask, CV, random_state=seed
    )

    log_df = objective.log.as_dataframe()
    log_df["best_cost_so_far"] = log_df["cost"].cummin()

    return {
        "seed": seed,
        "best_cost": float(best_cost),
        "best_mask": best_mask,
        "best_features": [FEATURE_COLS[i] for i, b in enumerate(best_mask) if b == 1],
        "n_features": int(best_mask.sum()),
        "best_metrics_cv": best_metrics,
        "n_evaluations": int(objective.log.eval_counter),
        "n_effective_trains": int(objective.effective_trains),
        "total_time_s": total_time,
        "log": log_df,
        "optimizer": optimizer,
        "objective": objective,
    }


print("Clases de objetivo y runner definidas.")


---
# 9. Experimentos y estabilidad

## Prueba piloto de tiempos

Antes del barrido multi-semilla se estima el tiempo de una evaluación de función objetivo (CV 3-fold de un Random Forest sobre un subconjunto aleatorio). Con ello se justifica el presupuesto final.

## Protocolo

1. Misma partición de desarrollo para todas las corridas.
2. Misma CV (`StratifiedKFold` con `GLOBAL_SEED`).
3. Semillas distintas solo para la estocasticidad de PSO (y `random_state` del RF dentro de cada corrida).
4. **No** se usa el test para elegir la mejor ejecución.


In [ ]:
# Prueba piloto: tiempo de una evaluación efectiva
rng = np.random.default_rng(GLOBAL_SEED)
pilot_mask = rng.integers(0, 2, size=N_FEATURES)
if pilot_mask.sum() == 0:
    pilot_mask[0] = 1

t0 = time.perf_counter()
pilot = evaluate_feature_subset(X_dev, y_dev, pilot_mask, CV, random_state=GLOBAL_SEED)
t_pilot = time.perf_counter() - t0

print("=== Prueba piloto ===")
print(f"Atributos en máscara piloto : {int(pilot_mask.sum())}")
print(f"Tiempo 1 evaluación FO (s)  : {t_pilot:.3f}")
print(f"F1-macro CV piloto          : {pilot['f1_macro_mean']:.4f}")
print(
    f"Tiempo estimado FULL / seed : {t_pilot * EXPECTED_EVALS:.1f} s "
    f"(~{t_pilot * EXPECTED_EVALS / 60:.1f} min) [orden de magnitud; caché puede reducir]"
)
print(
    f"Tiempo estimado {len(PSO_SEEDS)} semillas: "
    f"~{t_pilot * EXPECTED_EVALS * len(PSO_SEEDS) / 60:.1f} min"
)


In [ ]:
# Ejecución multi-semilla de Binary PSO
print(f"Iniciando Binary PSO en modo {EXPERIMENT_MODE}...")
print(f"Semillas: {PSO_SEEDS} | Evaluaciones esperadas/semilla: {EXPECTED_EVALS}\n")

pso_runs: List[Dict[str, Any]] = []
all_eval_logs: List[pd.DataFrame] = []

for seed in PSO_SEEDS:
    print(f"--- Ejecución seed={seed} ---")
    run = run_binary_pso(seed)
    pso_runs.append(run)
    all_eval_logs.append(run["log"])
    print(
        f"  best_cost={run['best_cost']:.6f} | n_feat={run['n_features']} | "
        f"F1_cv={run['best_metrics_cv']['f1_macro_mean']:.4f} | "
        f"evals={run['n_evaluations']} | trains={run['n_effective_trains']} | "
        f"time={run['total_time_s']:.1f}s"
    )
    print(f"  features: {run['best_features']}")

evals_df = pd.concat(all_eval_logs, ignore_index=True)
evals_df.to_csv(DIRS["csv"] / "pso_todas_las_evaluaciones.csv", index=False)

runs_summary = pd.DataFrame([
    {
        "seed": r["seed"],
        "best_cost": r["best_cost"],
        "n_features": r["n_features"],
        "f1_macro_cv": r["best_metrics_cv"]["f1_macro_mean"],
        "balanced_accuracy_cv": r["best_metrics_cv"]["balanced_accuracy_mean"],
        "roc_auc_cv": r["best_metrics_cv"]["roc_auc_mean"],
        "n_evaluations": r["n_evaluations"],
        "n_effective_trains": r["n_effective_trains"],
        "total_time_s": r["total_time_s"],
        "time_per_eval_s": r["total_time_s"] / max(r["n_evaluations"], 1),
        "selection_binary": "b" + "".join(str(int(b)) for b in r["best_mask"]),
        "selected_features": "|".join(r["best_features"]),
    }
    for r in pso_runs
])
runs_summary.to_csv(DIRS["csv"] / "pso_resumen_ejecuciones.csv", index=False)

print("\n=== Resumen entre ejecuciones ===")
display(runs_summary.round(4))

if len(runs_summary) > 1:
    print(
        "\nMedia ± std del mejor costo:",
        f"{runs_summary['best_cost'].mean():.4f} ± {runs_summary['best_cost'].std():.4f}",
    )
    print(
        "Media ± std de F1-macro CV:",
        f"{runs_summary['f1_macro_cv'].mean():.4f} ± {runs_summary['f1_macro_cv'].std():.4f}",
    )
    print(
        "Media ± std de n_features:",
        f"{runs_summary['n_features'].mean():.2f} ± {runs_summary['n_features'].std():.2f}",
    )
else:
    print("\n(Modo TEST: una sola ejecución; la variabilidad multi-semilla requiere FULL.)")


In [ ]:
# Selección de la mejor ejecución SOLO con desarrollo/CV (no test)
winner_row = runs_summary.sort_values(
    by=["best_cost", "f1_macro_cv", "n_features"],
    ascending=[True, False, True],
).iloc[0]
best_run = next(r for r in pso_runs if r["seed"] == int(winner_row["seed"]))

SELECTED_MASK = best_run["best_mask"].astype(bool)
SELECTED_FEATURES = best_run["best_features"]

print("=== Mejor ejecución (según CV / costo, sin mirar test) ===")
print(f"Semilla           : {best_run['seed']}")
print(f"Costo             : {best_run['best_cost']:.6f}")
print(f"F1-macro CV       : {best_run['best_metrics_cv']['f1_macro_mean']:.4f}")
print(f"n_features        : {best_run['n_features']}")
print(f"Atributos         : {SELECTED_FEATURES}")
print(f"Máscara binaria   : {''.join(str(int(b)) for b in best_run['best_mask'])}")

freq = Counter()
for r in pso_runs:
    for f in r["best_features"]:
        freq[f] += 1
freq_df = pd.DataFrame(
    [
        {
            "feature": f,
            "count": freq.get(f, 0),
            "frequency": freq.get(f, 0) / len(pso_runs),
        }
        for f in FEATURE_COLS
    ]
).sort_values("count", ascending=False)
freq_df.to_csv(DIRS["csv"] / "atributos_frecuencia_entre_semillas.csv", index=False)
print("\nFrecuencia de selección (mejor de cada semilla):")
display(freq_df)


---
# 10. Análisis de convergencia

Se analiza la **misma trayectoria** (mejor costo acumulado vs. número de evaluación) de la mejor ejecución (y de cada semilla).  
Los puntos de las evaluaciones **10, 20 y 30** pertenecen a esa trayectoria única — no a experimentos separados.

En modo TEST, si el presupuesto es menor a 30, se reportan los puntos disponibles y se advierte la limitación.


In [ ]:
def plot_convergence(log_df: pd.DataFrame, seed: int, outpath: Path) -> pd.DataFrame:
    """Grafica convergencia y extrae checkpoints 10/20/30 de la misma trayectoria."""
    df = log_df.copy()
    checkpoints = [10, 20, 30]
    rows = []
    for e in checkpoints:
        if (df["eval_id"] >= e).any():
            sub = df[df["eval_id"] <= e]
            rows.append({
                "seed": seed,
                "evaluacion": e,
                "best_cost": float(sub["best_cost_so_far"].iloc[-1]),
                "cost_at_eval": float(sub["cost"].iloc[-1]),
                "n_features_at_eval": int(sub["n_features"].iloc[-1]),
                "disponible": True,
            })
        else:
            rows.append({
                "seed": seed,
                "evaluacion": e,
                "best_cost": np.nan,
                "cost_at_eval": np.nan,
                "n_features_at_eval": np.nan,
                "disponible": False,
            })
    chk = pd.DataFrame(rows)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(df["eval_id"], df["cost"], alpha=0.35, label="Costo por evaluación", color="#9ecae1")
    ax.plot(
        df["eval_id"], df["best_cost_so_far"], linewidth=2.2,
        label="Mejor costo acumulado", color="#08519c",
    )
    for e in checkpoints:
        if e <= df["eval_id"].max():
            y = df.loc[df["eval_id"] == e, "best_cost_so_far"].values[0]
            ax.scatter([e], [y], s=70, zorder=5)
            ax.annotate(
                f"eval {e}\n{y:.4f}", (e, y),
                textcoords="offset points", xytext=(6, 8), fontsize=8,
            )
    ax.set_xlabel("Número de evaluación de la función objetivo")
    ax.set_ylabel("Costo (minimizar)")
    ax.set_title(f"Convergencia Binary PSO — semilla {seed} (misma trayectoria)")
    ax.legend()
    fig.tight_layout()
    fig.savefig(outpath, dpi=150, bbox_inches="tight")
    plt.show()
    return chk


checkpoint_tables = []
for r in pso_runs:
    out = DIRS["convergence"] / f"convergencia_seed_{r['seed']}.png"
    chk = plot_convergence(r["log"], r["seed"], out)
    checkpoint_tables.append(chk)
    print(f"Checkpoints semilla {r['seed']}:")
    display(chk)

checkpoints_df = pd.concat(checkpoint_tables, ignore_index=True)
checkpoints_df.to_csv(DIRS["csv"] / "pso_checkpoints_10_20_30.csv", index=False)

blog = best_run["log"]
print("\n=== Lectura de convergencia (mejor semilla) ===")
print(f"Evaluaciones totales en trayectoria: {blog['eval_id'].max()}")
print(f"Costo inicial (eval 1): {blog.loc[blog['eval_id']==1, 'cost'].values[0]:.6f}")
print(f"Mejor costo final     : {blog['best_cost_so_far'].iloc[-1]:.6f}")
improvements = blog["best_cost_so_far"].diff().fillna(0)
n_improve = int((improvements < 0).sum())
print(f"Actualizaciones del mejor global: {n_improve}")
if n_improve <= 1:
    print("Se observa poca mejora progresiva (posible convergencia temprana o estancamiento).")
else:
    print("Se observan mejoras progresivas a lo largo del presupuesto.")


---
# 11. Evaluación final y comparación

Protocolo (sin fuga de información):

1. Elegir la máscara de atributos con desarrollo/CV (ya hecho).
2. Reentrenar RF baseline (11 attrs) en **todo el desarrollo**.
3. Reentrenar RF + PSO (subconjunto) en **todo el desarrollo**.
4. Evaluar **una sola vez** ambos modelos en el test reservado.


In [ ]:
def fit_and_eval_test(
    X_tr: pd.DataFrame,
    y_tr: np.ndarray,
    X_te: pd.DataFrame,
    y_te: np.ndarray,
    feature_names: List[str],
    random_state: int = GLOBAL_SEED,
) -> Dict[str, Any]:
    pipe = make_pipeline(random_state=random_state)
    t0 = time.perf_counter()
    pipe.fit(X_tr[feature_names], y_tr)
    fit_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    proba = pipe.predict_proba(X_te[feature_names])[:, 1]
    pred = (proba >= 0.5).astype(int)
    eval_time = time.perf_counter() - t1

    return {
        "n_features": len(feature_names),
        "features": feature_names,
        "f1_macro": f1_score(y_te, pred, average="macro"),
        "f1_binary": f1_score(y_te, pred, average="binary"),
        "balanced_accuracy": balanced_accuracy_score(y_te, pred),
        "precision_macro": precision_score(y_te, pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_te, pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_te, pred),
        "roc_auc": roc_auc_score(y_te, proba),
        "fit_time_s": fit_time,
        "eval_time_s": eval_time,
        "y_true": y_te,
        "y_pred": pred,
        "y_proba": proba,
        "model": pipe,
    }


baseline_test = fit_and_eval_test(
    X_dev, y_dev, X_test, y_test, FEATURE_COLS, random_state=GLOBAL_SEED
)
pso_test = fit_and_eval_test(
    X_dev, y_dev, X_test, y_test, SELECTED_FEATURES, random_state=GLOBAL_SEED
)

comparison_rows = []
metric_keys = [
    ("F1-score (macro)", "f1_macro"),
    ("F1-score (binary)", "f1_binary"),
    ("Balanced accuracy", "balanced_accuracy"),
    ("Precision (macro)", "precision_macro"),
    ("Recall (macro)", "recall_macro"),
    ("Accuracy", "accuracy"),
    ("ROC-AUC", "roc_auc"),
    ("Cantidad de atributos", "n_features"),
    ("Tiempo de entrenamiento (s)", "fit_time_s"),
    ("Tiempo de evaluación (s)", "eval_time_s"),
]
for label, key in metric_keys:
    comparison_rows.append({
        "Métrica": label,
        "Random Forest Base": baseline_test[key],
        "Random Forest + PSO": pso_test[key],
    })

comparison_df = pd.DataFrame(comparison_rows)
display_cmp = comparison_df.copy()
for col in ["Random Forest Base", "Random Forest + PSO"]:
    display_cmp[col] = display_cmp.apply(
        lambda r: (
            f"{r[col]:.0f}" if r["Métrica"] == "Cantidad de atributos"
            else f"{r[col]:.4f}"
        ),
        axis=1,
    )
print("=== Comparación en TEST (una sola evaluación) ===")
display(display_cmp)

comparison_df.to_csv(DIRS["csv"] / "comparacion_baseline_vs_pso_test.csv", index=False)

print("\nAtributos seleccionados por PSO:")
print(SELECTED_FEATURES)
print(
    f"\nReducción: {11 - len(SELECTED_FEATURES)} atributos eliminados "
    f"({100 * (11 - len(SELECTED_FEATURES)) / 11:.1f}%)."
)


In [ ]:
# Matrices de confusión y ROC
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, res, title in [
    (axes[0], baseline_test, "RF Baseline (11 attrs)"),
    (axes[1], pso_test, f"RF + PSO ({pso_test['n_features']} attrs)"),
]:
    cm = confusion_matrix(res["y_true"], res["y_pred"])
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues", ax=ax,
        xticklabels=["Pred 0", "Pred 1"], yticklabels=["Real 0", "Real 1"],
    )
    ax.set_title(title)
fig.suptitle("Matrices de confusión en TEST", fontsize=13)
fig.tight_layout()
fig.savefig(DIRS["evaluation"] / "matrices_confusion_test.png", dpi=150, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for res, label, color in [
    (baseline_test, f"Baseline AUC={baseline_test['roc_auc']:.3f}", "#2166ac"),
    (pso_test, f"PSO AUC={pso_test['roc_auc']:.3f}", "#b2182b"),
]:
    fpr, tpr, _ = roc_curve(res["y_true"], res["y_proba"])
    ax.plot(fpr, tpr, label=label, color=color, linewidth=2)
ax.plot([0, 1], [0, 1], "k--", alpha=0.4)
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("Curvas ROC en TEST")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(DIRS["evaluation"] / "roc_test.png", dpi=150, bbox_inches="tight")
plt.show()

plot_metrics = ["f1_macro", "balanced_accuracy", "roc_auc"]
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(plot_metrics))
w = 0.35
ax.bar(x - w/2, [baseline_test[m] for m in plot_metrics], w, label="Baseline", color="#2166ac")
ax.bar(x + w/2, [pso_test[m] for m in plot_metrics], w, label="RF+PSO", color="#b2182b")
ax.set_xticks(x)
ax.set_xticklabels(plot_metrics)
ax.set_ylim(0, 1.05)
ax.set_ylabel("Puntuación")
ax.set_title("Comparación de métricas en TEST")
ax.legend()
fig.tight_layout()
fig.savefig(DIRS["evaluation"] / "comparacion_metricas_test.png", dpi=150, bbox_inches="tight")
plt.show()

print("Reporte de clasificación — Baseline:")
print(classification_report(baseline_test["y_true"], baseline_test["y_pred"], digits=4))
print("Reporte de clasificación — RF+PSO:")
print(classification_report(pso_test["y_true"], pso_test["y_pred"], digits=4))


In [ ]:
# Frecuencia de atributos — gráfico
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=freq_df, x="feature", y="frequency", ax=ax, color="#4393c3")
ax.set_title("Frecuencia de selección de atributos entre ejecuciones independientes")
ax.set_xlabel("Atributo")
ax.set_ylabel("Frecuencia relativa")
ax.set_ylim(0, 1.05)
plt.xticks(rotation=45, ha="right")
fig.tight_layout()
fig.savefig(DIRS["evaluation"] / "frecuencia_atributos.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 12. Costo computacional

Se utiliza `time.perf_counter()` (reloj monotónico) para:

- tiempo por evaluación de FO;
- tiempo total de PSO por semilla;
- tiempo de entrenamiento/evaluación de baseline y modelo optimizado;
- número de evaluaciones efectivas y de entrenamientos CV.

La pregunta central: **¿la reducción de variables compensa el costo de la búsqueda?**  
Distinguimos *tiempo de búsqueda* (PSO) de *tiempo del modelo final* (entrenamiento único sobre desarrollo).


In [ ]:
cost_rows = []
for r in pso_runs:
    log = r["log"]
    cost_rows.append({
        "seed": r["seed"],
        "total_pso_time_s": r["total_time_s"],
        "n_evaluations": r["n_evaluations"],
        "n_effective_trains": r["n_effective_trains"],
        "mean_eval_time_s": log["time_s"].mean(),
        "std_eval_time_s": log["time_s"].std(),
        "sum_eval_time_s": log["time_s"].sum(),
        "n_cv_model_fits_approx": int(r["n_effective_trains"] * N_SPLITS),
    })

cost_df = pd.DataFrame(cost_rows)
cost_df.to_csv(DIRS["csv"] / "costo_computacional_pso.csv", index=False)
print("=== Costo por ejecución PSO ===")
display(cost_df.round(4))

print("\n=== Tiempos de modelos finales ===")
print(f"Baseline CV (desarrollo)     : {baseline_cv['total_time']:.3f} s")
print(f"Baseline fit desarrollo       : {baseline_test['fit_time_s']:.3f} s")
print(f"Baseline eval test            : {baseline_test['eval_time_s']:.3f} s")
print(f"PSO fit desarrollo            : {pso_test['fit_time_s']:.3f} s")
print(f"PSO eval test                 : {pso_test['eval_time_s']:.3f} s")
print(f"Tiempo total búsqueda PSO     : {sum(r['total_time_s'] for r in pso_runs):.1f} s")

search_time = sum(r["total_time_s"] for r in pso_runs)
final_saving = baseline_test["fit_time_s"] - pso_test["fit_time_s"]
print(
    f"\nInterpretación: la búsqueda costó {search_time:.1f}s; "
    f"el eventual ahorro de entrenamiento final es {final_saving:.4f}s. "
    "En este problema el valor de PSO es más de parsimonia/interpretabilidad "
    "que de ahorro de wall-clock del modelo final."
)


---
# 13. Discusión de resultados

Las respuestas siguientes se **generan a partir de los resultados reales** de la ejecución (no son texto fijo inventado).


In [ ]:
# Discusión basada en evidencias de la corrida actual
delta_f1 = pso_test["f1_macro"] - baseline_test["f1_macro"]
delta_bal = pso_test["balanced_accuracy"] - baseline_test["balanced_accuracy"]
n_removed = 11 - len(SELECTED_FEATURES)

if len(pso_runs) >= 3:
    masks = [r["best_mask"] for r in pso_runs]
    jaccards = []
    for i in range(len(masks)):
        for j in range(i + 1, len(masks)):
            inter = np.logical_and(masks[i], masks[j]).sum()
            union = np.logical_or(masks[i], masks[j]).sum()
            jaccards.append(inter / union if union else 1.0)
    mean_j = float(np.mean(jaccards))
    stable = mean_j >= 0.5
else:
    mean_j = np.nan
    stable = False

blog = best_run["log"]
n_updates = int((blog["best_cost_so_far"].diff().fillna(0) < 0).sum())

interp = (
    "mejora" if delta_f1 > 0.005
    else ("empeora" if delta_f1 < -0.005 else "queda esencialmente empatado")
)
jacc_txt = f"{mean_j:.3f}" if not np.isnan(mean_j) else "N/A (modo TEST / <3 semillas)"
stab_txt = "relativamente estable" if stable else "variable / evidencia insuficiente"
stag_txt = (
    "Sí hay indicios de estancamiento (pocas mejoras)."
    if n_updates <= 1
    else "Hubo mejoras a lo largo del presupuesto; el estancamiento, si existe, es parcial."
)
total_search = sum(r["total_time_s"] for r in pso_runs)

print("## Respuestas a las preguntas de discusión\n")
print("1. **¿PSO mejoró el rendimiento predictivo en test?**")
print(f"   - Δ F1-macro (PSO − Baseline) = {delta_f1:+.4f}")
print(f"   - Δ Balanced accuracy = {delta_bal:+.4f}")
print(f"   - Interpretación: {interp} respecto al baseline.\n")
print("2. **¿Cuántos atributos eliminó?**")
print(f"   - Eliminó {n_removed} de 11 ({100 * n_removed / 11:.1f}%).")
print(f"   - Subconjunto final ({len(SELECTED_FEATURES)}): {SELECTED_FEATURES}\n")
print("3. **¿Cuáles se seleccionaron con mayor frecuencia?**")
display(freq_df.head(5))
print("\n4. **¿El subconjunto fue estable entre ejecuciones?**")
print(f"   - Jaccard medio entre mejores máscaras: {jacc_txt}")
print(f"   - Lectura: {stab_txt}\n")
print("5. **¿Cuánto costó computacionalmente?**")
print(f"   - Tiempo total de búsqueda: {total_search:.1f} s")
print(f"   - Evaluaciones totales: {int(evals_df.shape[0])}")
print(f"   - Entrenamientos efectivos (FO con RF): {int(evals_df['trained'].sum())}\n")
print("6. **¿Hubo estancamiento?**")
print(f"   - Actualizaciones del mejor global en la mejor trayectoria: {n_updates}")
print(f"   - {stag_txt}\n")
print("7. **¿Conviene utilizar PSO en este problema?**")
print("   - Conviene si se valora parsimonia/interpretabilidad y el Δ de métricas en test no es adverso.")
print("   - Menos conveniente si el único objetivo es maximizar métricas y el baseline ya es competitivo")
print("     con 11 variables bajo un presupuesto corto (~30 evals).\n")
print("8. **Limitaciones**")
print("   - Espacio 2^11 explorado solo parcialmente bajo presupuesto cerrado.")
print("   - Hiperparámetros de RF fijos (por diseño del TP).")
print("   - Binarización pierde ordinalidad de quality.")
print(f"   - Modo actual: {EXPERIMENT_MODE} (si es TEST, no generalizar conclusiones).")
print("   - Una sola partición train/test; la varianza de split no se estima exhaustivamente.")


---
# 14. Conclusiones

Las conclusiones se redactan de forma automática a partir de los resultados de **esta** ejecución, para evitar afirmaciones no respaldadas.


In [ ]:
delta_f1_test = pso_test["f1_macro"] - baseline_test["f1_macro"]
conclusiones = (
    f"## Conclusiones académicas (evidencia de la corrida {EXPERIMENT_MODE})\n\n"
    f"**Cumplimiento de objetivos.** Se construyó un protocolo reproducible de clasificación "
    f"binaria sobre Wine Quality (tintos, UCI), con baseline Random Forest de 11 atributos y "
    f"selección mediante Binary PSO bajo presupuesto controlado ({EXPECTED_EVALS} "
    f"evaluaciones/ejecución, CV estratificada de {N_SPLITS} folds). El test se evaluó una "
    f"sola vez tras fijar el subconjunto.\n\n"
    f"**Hallazgos principales.**\n"
    f"- Baseline (CV): F1-macro = {baseline_cv['f1_macro_mean']:.4f}\n"
    f"- Mejor PSO (CV): F1-macro = {best_run['best_metrics_cv']['f1_macro_mean']:.4f}, "
    f"costo = {best_run['best_cost']:.6f}, n_features = {best_run['n_features']}\n"
    f"- Test — Baseline F1-macro = {baseline_test['f1_macro']:.4f} | "
    f"RF+PSO F1-macro = {pso_test['f1_macro']:.4f} (Δ = {delta_f1_test:+.4f})\n"
    f"- Atributos retenidos: {SELECTED_FEATURES}\n\n"
    f"**Beneficios.** Reducción a {len(SELECTED_FEATURES)} variables; evidencia exportable "
    f"(CSV/figuras) para artículo y póster; protocolo sin fuga de información.\n\n"
    f"**Limitaciones.** Presupuesto metaheurístico limitado frente a 2048 subconjuntos; "
    f"etiqueta binaria simplifica un problema ordinal; resultados de modo TEST no son "
    f"definitivos.\n\n"
    f"**Trabajos futuros.** Aumentar presupuesto / semillas; comparar con búsqueda "
    f"exhaustiva o wrappers alternativos; incorporar importancia de Gini/permutación como "
    f"baseline de selección; estudiar el problema como ordinal/regresión; optimizar "
    f"hiperparámetros *después* de fijar atributos (fuera del alcance de este TP).\n"
)
print(conclusiones)

with open(DIRS["logs"] / "conclusiones.txt", "w", encoding="utf-8") as f:
    f.write(conclusiones)


---
# 15. Exportación de resultados

Se exportan automáticamente CSV, figuras y un manifiesto JSON con rutas.  
En Colab, los archivos quedan bajo `/content/resultados/`.


In [ ]:
# Exportaciones adicionales / manifiesto
selected_attrs_df = pd.DataFrame({
    "feature": FEATURE_COLS,
    "selected_in_best_run": [f in SELECTED_FEATURES for f in FEATURE_COLS],
    "frequency_across_seeds": [freq.get(f, 0) / len(pso_runs) for f in FEATURE_COLS],
})
selected_attrs_df.to_csv(DIRS["csv"] / "atributos_seleccionados.csv", index=False)

summary_json = {
    "mode": EXPERIMENT_MODE,
    "dataset_url": DATA_URL,
    "n_samples_total": int(len(df_raw)),
    "n_dev": int(len(X_dev)),
    "n_test": int(len(X_test)),
    "threshold_quality": QUALITY_THRESHOLD,
    "lambda_penalty": LAMBDA_PENALTY,
    "rf_params": RF_PARAMS,
    "pso": {
        "n_particles": PSO_N_PARTICLES,
        "iters": PSO_ITERS,
        "expected_evals": EXPECTED_EVALS,
        "options": PSO_OPTIONS,
        "seeds": PSO_SEEDS,
    },
    "baseline_cv": {
        k: (float(v) if isinstance(v, (float, np.floating, int, np.integer)) else v)
        for k, v in baseline_cv.items()
        if k != "wall_time"
    },
    "best_run": {
        "seed": int(best_run["seed"]),
        "best_cost": float(best_run["best_cost"]),
        "n_features": int(best_run["n_features"]),
        "features": SELECTED_FEATURES,
        "selection_binary": "b" + "".join(str(int(b)) for b in best_run["best_mask"]),
        "f1_macro_cv": float(best_run["best_metrics_cv"]["f1_macro_mean"]),
    },
    "test_metrics": {
        "baseline": {
            k: float(baseline_test[k])
            for k in [
                "f1_macro", "balanced_accuracy", "precision_macro", "recall_macro",
                "accuracy", "roc_auc", "n_features", "fit_time_s", "eval_time_s",
            ]
        },
        "pso": {
            k: float(pso_test[k])
            for k in [
                "f1_macro", "balanced_accuracy", "precision_macro", "recall_macro",
                "accuracy", "roc_auc", "n_features", "fit_time_s", "eval_time_s",
            ]
        },
    },
    "env": ENV_INFO,
}

with open(DIRS["logs"] / "resumen_experimento.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2, ensure_ascii=False)

artifacts = sorted(
    [str(p.relative_to(DIRS["root"])) for p in DIRS["root"].rglob("*") if p.is_file()]
)
manifest = pd.DataFrame({"artifact": artifacts})
manifest.to_csv(DIRS["csv"] / "manifiesto_artefactos.csv", index=False)

print(f"Resultados exportados en: {DIRS['root'].resolve()}")
print(f"Total de archivos: {len(artifacts)}")
print("\nPrincipales CSV:")
for p in sorted(DIRS["csv"].glob("*.csv")):
    print(f"  - {p}")
print("\nFiguras:")
for p in sorted(DIRS["figures"].rglob("*.png")):
    print(f"  - {p}")


---
# Referencias

1. P. Cortez, A. Cerdeira, F. Almeida, T. Matos y J. Reis, "Modeling wine preferences by data mining from physicochemical properties," *Decision Support Systems*, vol. 47, no. 4, pp. 547–553, 2009.
2. UCI Machine Learning Repository — Wine Quality: https://archive.ics.uci.edu/dataset/186/wine+quality
3. L. Breiman, "Random Forests," *Machine Learning*, vol. 45, pp. 5–32, 2001.
4. J. Kennedy y R. Eberhart, "Particle swarm optimization," en *Proc. ICNN'95*, 1995, pp. 1942–1948.
5. J. Kennedy y R. C. Eberhart, "A discrete binary version of the particle swarm algorithm," en *Proc. IEEE SMC*, 1997.
6. L. J. V. Miranda, "PySwarms: a research toolkit for Particle Swarm Optimization in Python," *JOSS*, 2018. https://pyswarms.readthedocs.io/
7. F. Pedregosa et al., "Scikit-learn: Machine Learning in Python," *JMLR*, vol. 12, pp. 2825–2830, 2011.

---

## Nota final de uso

1. Subir este `.ipynb` a Google Colab.
2. Dejar `EXPERIMENT_MODE = "TEST"` para una verificación rápida.
3. Cambiar a `EXPERIMENT_MODE = "FULL"` para el experimento del informe (≈ 30 evaluaciones × 5 semillas).
4. *Entorno de ejecución → Ejecutar todo*.
5. Descargar la carpeta `resultados/` para el artículo IEEE y el póster.

**Advertencia:** si el modo activo es `TEST`, no presentar esas métricas como resultados definitivos del Trabajo Práctico.
